# Read vs Spontaneous — Evaluation Notebook

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    confusion_matrix, precision_recall_curve,
    roc_curve, auc, classification_report
)

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

## 1 — Load your data

In [ ]:
# ── YOUR INPUTS ──────────────────────────────────────────────
PREDICTIONS_JSON = "outputs/results_audios2.json"   # path to predictions JSON
CURRENT_THRESHOLD = 0.50                             # threshold used during inference
# ─────────────────────────────────────────────────────────────

# Build your GT dataframe however you have it.
# It needs two columns: 'filename' and 'gt_label' (1=read, 0=spontaneous)
# Example — replace this block with however you load your labels:
gt_df = pd.DataFrame({
    'filename': ['file001.wav', 'file002.wav'],   # replace with your filenames
    'gt_label': [1, 0],                           # 1 = read, 0 = spontaneous
})

gt_df.head()

In [ ]:
# Load predictions JSON and merge with GT
with open(PREDICTIONS_JSON) as f:
    raw = json.load(f)

pred_df = pd.DataFrame([{
    'filename':           r['filename'],
    'duration_sec':       r['duration_sec'],
    'read_ratio':         r['read_ratio'],
    'overall_confidence': r['overall_confidence'],
    'overall_label':      r['overall_label'],
    'n_windows':          len(r.get('window_predictions', [])),
    'n_segments':         len(r.get('segments', [])),
    'processing_time_sec': r.get('processing_time_sec', 0),
} for r in raw])

df = pd.merge(gt_df, pred_df, on='filename', how='inner')

# Derived columns
df['pred_label']     = (df['read_ratio'] >= CURRENT_THRESHOLD).astype(int)
df['correct']        = (df['pred_label'] == df['gt_label']).astype(int)
df['gt_label_name']  = df['gt_label'].map({1: 'read', 0: 'spontaneous'})
df['pred_label_name']= df['pred_label'].map({1: 'read', 0: 'spontaneous'})
df['outcome'] = df.apply(lambda r: (
    'TP' if r.gt_label==1 and r.pred_label==1 else
    'TN' if r.gt_label==0 and r.pred_label==0 else
    'FN' if r.gt_label==1 and r.pred_label==0 else 'FP'
), axis=1)

print(f"Matched {len(df)} files  |  "
      f"Read (GT): {df.gt_label.sum()}  |  "
      f"Spontaneous (GT): {(df.gt_label==0).sum()}")
df.head()

## 2 — Core metrics at current threshold

In [ ]:
y_true = df['gt_label'].values
y_pred = df['pred_label'].values
read_ratios = df['read_ratio'].values

p  = precision_score(y_true, y_pred, zero_division=0)
r  = recall_score(y_true, y_pred, zero_division=0)
f1 = f1_score(y_true, y_pred, zero_division=0)
cm = confusion_matrix(y_true, y_pred)

print(f"Threshold : {CURRENT_THRESHOLD}")
print(f"Precision : {p:.1%}")
print(f"Recall    : {r:.1%}")
print(f"F1        : {f1:.1%}")
print()
print(classification_report(y_true, y_pred, target_names=['spontaneous','read']))
print("Outcome counts:")
print(df['outcome'].value_counts())

## 3 — Threshold sweep — find optimal

In [ ]:
precision_curve, recall_curve, thresholds = precision_recall_curve(y_true, read_ratios)

sweep = []
for t in np.arange(0.10, 0.91, 0.01):
    preds = (read_ratios >= t).astype(int)
    sweep.append({
        'threshold': round(t, 2),
        'precision': precision_score(y_true, preds, zero_division=0),
        'recall':    recall_score(y_true, preds, zero_division=0),
        'f1':        f1_score(y_true, preds, zero_division=0),
    })

sweep_df = pd.DataFrame(sweep)

# Best threshold: max recall while precision >= 90%
eligible = sweep_df[sweep_df['precision'] >= 0.90]
if not eligible.empty:
    best = eligible.loc[eligible['recall'].idxmax()]
    print(f"Optimal threshold (P >= 90%): {best.threshold}")
    print(f"  Precision: {best.precision:.1%}  Recall: {best.recall:.1%}  F1: {best.f1:.1%}")
else:
    best = sweep_df.loc[sweep_df['f1'].idxmax()]
    print(f"No threshold achieves P>=90%. Best F1 threshold: {best.threshold}")
    print(f"  Precision: {best.precision:.1%}  Recall: {best.recall:.1%}  F1: {best.f1:.1%}")

OPTIMAL_THRESHOLD = best.threshold
sweep_df

## 4 — Plots

In [ ]:
# ── Plot 1: Precision / Recall / F1 vs threshold ──────────────
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(sweep_df['threshold'], sweep_df['precision'], label='Precision', lw=2)
ax.plot(sweep_df['threshold'], sweep_df['recall'],    label='Recall',    lw=2)
ax.plot(sweep_df['threshold'], sweep_df['f1'],        label='F1',        lw=2, linestyle='--')
ax.axvline(CURRENT_THRESHOLD,  color='gray',   linestyle=':', lw=1.5, label=f'Current ({CURRENT_THRESHOLD})')
ax.axvline(OPTIMAL_THRESHOLD,  color='crimson',linestyle=':', lw=1.5, label=f'Optimal ({OPTIMAL_THRESHOLD})')
ax.axhline(0.90, color='black', linestyle=':', lw=1, alpha=0.4, label='90% line')
ax.set_xlabel('Threshold')
ax.set_ylabel('Score')
ax.set_title('Precision / Recall / F1 across thresholds')
ax.legend()
ax.set_xlim(0.10, 0.90)
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

In [ ]:
# ── Plot 2: Precision-Recall curve ───────────────────────────
fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(recall_curve, precision_curve, lw=2, color='steelblue')
ax.fill_between(recall_curve, precision_curve, alpha=0.08, color='steelblue')

# Mark current and optimal points
for thresh, label, color in [
    (CURRENT_THRESHOLD, f'Current ({CURRENT_THRESHOLD})', 'gray'),
    (OPTIMAL_THRESHOLD, f'Optimal ({OPTIMAL_THRESHOLD})', 'crimson'),
]:
    preds = (read_ratios >= thresh).astype(int)
    pt_p = precision_score(y_true, preds, zero_division=0)
    pt_r = recall_score(y_true, preds, zero_division=0)
    ax.scatter(pt_r, pt_p, s=80, color=color, zorder=5, label=label)

ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Precision-Recall curve')
ax.legend()
ax.set_xlim(0, 1.05)
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

In [ ]:
# ── Plot 3: Confusion matrix ──────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for ax, thresh, title in [
    (axes[0], CURRENT_THRESHOLD, f'Current threshold ({CURRENT_THRESHOLD})'),
    (axes[1], OPTIMAL_THRESHOLD, f'Optimal threshold ({OPTIMAL_THRESHOLD})'),
]:
    preds = (read_ratios >= thresh).astype(int)
    cm = confusion_matrix(y_true, preds)
    sns.heatmap(
        cm, annot=True, fmt='d', ax=ax,
        xticklabels=['spontaneous','read'],
        yticklabels=['spontaneous','read'],
        cmap='Blues', cbar=False
    )
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
    ax.set_title(title)

plt.tight_layout()
plt.show()

In [ ]:
# ── Plot 4: read_ratio distribution by GT class ───────────────
fig, ax = plt.subplots(figsize=(10, 4))

for label, color in [('read', 'tomato'), ('spontaneous', 'steelblue')]:
    subset = df[df['gt_label_name'] == label]['read_ratio']
    ax.hist(subset, bins=30, alpha=0.6, color=color, label=label, density=True)

ax.axvline(CURRENT_THRESHOLD,  color='gray',   linestyle='--', lw=1.5, label=f'Current ({CURRENT_THRESHOLD})')
ax.axvline(OPTIMAL_THRESHOLD,  color='crimson',linestyle='--', lw=1.5, label=f'Optimal ({OPTIMAL_THRESHOLD})')
ax.set_xlabel('read_ratio')
ax.set_ylabel('Density')
ax.set_title('read_ratio distribution — GT read vs GT spontaneous')
ax.legend()
plt.tight_layout()
plt.show()

# This plot is the most important one — if the two distributions heavily overlap
# near the threshold, no threshold tuning will fix the problem. You need a better model.
# If they are well separated, threshold tuning alone can recover most of the recall gap.

In [ ]:
# ── Plot 5: Confidence distribution by outcome ────────────────
fig, ax = plt.subplots(figsize=(10, 4))

colors = {'TP': 'green', 'TN': 'steelblue', 'FN': 'tomato', 'FP': 'orange'}
for outcome, color in colors.items():
    subset = df[df['outcome'] == outcome]['overall_confidence']
    if len(subset) > 0:
        ax.hist(subset, bins=20, alpha=0.6, color=color,
                label=f'{outcome} (n={len(subset)})', density=True)

ax.set_xlabel('overall_confidence')
ax.set_ylabel('Density')
ax.set_title('Model confidence by outcome (FN = missed reads, FP = false alarms)')
ax.legend()
plt.tight_layout()
plt.show()

# If FN confidence is HIGH, the model is confidently wrong — harder to fix with threshold.
# If FN confidence is LOW (near 0.5), the model is uncertain — threshold tuning helps.

In [ ]:
# ── Plot 6: read_ratio vs confidence, coloured by outcome ─────
fig, ax = plt.subplots(figsize=(9, 6))

colors = {'TP': 'green', 'TN': 'steelblue', 'FN': 'tomato', 'FP': 'orange'}
for outcome, color in colors.items():
    subset = df[df['outcome'] == outcome]
    ax.scatter(
        subset['read_ratio'], subset['overall_confidence'],
        c=color, label=f'{outcome} (n={len(subset)})',
        alpha=0.7, s=60, edgecolors='white', linewidths=0.5
    )

ax.axvline(CURRENT_THRESHOLD,  color='gray',   linestyle='--', lw=1.5, label=f'Current ({CURRENT_THRESHOLD})')
ax.axvline(OPTIMAL_THRESHOLD,  color='crimson',linestyle='--', lw=1.5, label=f'Optimal ({OPTIMAL_THRESHOLD})')
ax.set_xlabel('read_ratio')
ax.set_ylabel('overall_confidence')
ax.set_title('read_ratio vs confidence — each dot is one file')
ax.legend()
plt.tight_layout()
plt.show()

# FN dots (red) clustered left of the threshold = lowering threshold will catch them
# FN dots (red) clustered far left with high confidence = model is fundamentally wrong
#   on those files — those are your hesitant readers that need training data

In [ ]:
# ── Plot 7: Duration vs read_ratio ───────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))

for outcome, color in colors.items():
    subset = df[df['outcome'] == outcome]
    ax.scatter(
        subset['duration_sec'], subset['read_ratio'],
        c=color, label=f'{outcome} (n={len(subset)})',
        alpha=0.7, s=60, edgecolors='white', linewidths=0.5
    )

ax.axhline(CURRENT_THRESHOLD,  color='gray',   linestyle='--', lw=1.5)
ax.axhline(OPTIMAL_THRESHOLD,  color='crimson',linestyle='--', lw=1.5)
ax.set_xlabel('Duration (seconds)')
ax.set_ylabel('read_ratio')
ax.set_title('Duration vs read_ratio — does file length affect accuracy?')
ax.legend()
plt.tight_layout()
plt.show()

## 5 — Error analysis: missed reads and false alarms

In [ ]:
missed_reads = df[df['outcome'] == 'FN'].sort_values('read_ratio', ascending=False)
false_alarms = df[df['outcome'] == 'FP'].sort_values('read_ratio', ascending=False)

print(f"=== MISSED READS (FN) — {len(missed_reads)} files ===")
print("These are read files the model called spontaneous.")
print("Files at the top have highest read_ratio — closest to threshold, easiest to recover.")
print("Files at the bottom have lowest read_ratio — model is confidently wrong on these.")
print()
display(missed_reads[[
    'filename','read_ratio','overall_confidence','duration_sec','n_windows'
]].reset_index(drop=True))

print(f"\n=== FALSE ALARMS (FP) — {len(false_alarms)} files ===")
print("These are spontaneous files the model called read.")
display(false_alarms[[
    'filename','read_ratio','overall_confidence','duration_sec','n_windows'
]].reset_index(drop=True))

In [ ]:
# Files recoverable by threshold tuning alone
# = FN files whose read_ratio falls between OPTIMAL and CURRENT threshold
recoverable = missed_reads[
    (missed_reads['read_ratio'] >= OPTIMAL_THRESHOLD) &
    (missed_reads['read_ratio'] < CURRENT_THRESHOLD)
]

hard_cases = missed_reads[missed_reads['read_ratio'] < OPTIMAL_THRESHOLD]

print(f"Recoverable by threshold tuning alone: {len(recoverable)} files")
print(f"Hard cases (model fundamentally wrong): {len(hard_cases)} files")
print()
print("Hard cases — these need training data with hesitant readers:")
display(hard_cases[['filename','read_ratio','overall_confidence']].reset_index(drop=True))

## 6 — Summary comparison table

In [ ]:
rows = []
for thresh in [CURRENT_THRESHOLD, OPTIMAL_THRESHOLD]:
    preds = (read_ratios >= thresh).astype(int)
    tp = int(((preds==1) & (y_true==1)).sum())
    tn = int(((preds==0) & (y_true==0)).sum())
    fp = int(((preds==1) & (y_true==0)).sum())
    fn = int(((preds==0) & (y_true==1)).sum())
    rows.append({
        'Threshold':  thresh,
        'Precision':  f"{precision_score(y_true, preds, zero_division=0):.1%}",
        'Recall':     f"{recall_score(y_true, preds, zero_division=0):.1%}",
        'F1':         f"{f1_score(y_true, preds, zero_division=0):.1%}",
        'TP': tp, 'TN': tn, 'FP': fp, 'FN': fn,
    })

summary = pd.DataFrame(rows)
summary.index = ['Current', 'Optimal']
display(summary)

## Notes on interpreting the plots

**Plot 4 (read_ratio distribution)** is the most diagnostic. If the two distributions (red=read, blue=spontaneous) heavily overlap near the threshold, no amount of threshold tuning will fix the problem — you need a better model or more training data covering the hard cases.

**Plot 5 (confidence by outcome)** tells you whether the model's errors are confident or uncertain. FN (missed reads) with HIGH confidence = model is fundamentally wrong on those files — these are your hesitant readers that need to be in training data. FN with LOW confidence = model is uncertain — threshold tuning can recover these.

**Plot 6 (scatter)** gives you the clearest view of where the threshold line should go. FN dots (red/tomato) clustered just left of the threshold = easy wins from lowering it. FN dots far left with high confidence = hard cases that need model improvement.